In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
# 1. Load Data
df_busco = pd.read_csv('../data/busco_enterobacterales_table.tsv', sep='\t')

In [33]:
# Sort by Complete%
df_busco_sorted = df_busco.sort_values('Complete%', ascending=True)


In [35]:
# Prepare Data for Stacked Bar
# We need columns: 'Single%', 'Dup%', 'Frag%', 'Miss%'
plot_data = df_busco_sorted[['Sample', 'Single%', 'Dup%', 'Frag%', 'Miss%']].set_index('Sample')

In [ ]:
# Define BUSCO standard colors
busco_colors = {
    'Single%': '#66BD63', # Light Green
    'Dup%': '#1A9850',    # Dark Green
    'Frag%': '#FFFFBF',   # Yellow
    'Miss%': '#D73027'    # Red
}

# Tall figure to accommodate all samples without label crowding
fig, ax = plt.subplots(figsize=(10, 12))

plot_data.plot(
    kind='barh', 
    stacked=True, 
    color=[busco_colors[col] for col in plot_data.columns],
    width=0.8,
    ax=ax
)

ax.set_title('Gene Content Completeness (BUSCO)', fontsize=14, fontweight='bold')
ax.set_xlabel('Percentage of BUSCO Genes')
ax.set_ylabel('Genome Assembly')
ax.set_xlim(0, 100)
ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='y', labelsize=6)

plt.tight_layout()
plt.savefig('Figure_BUSCO_StackedBar.tiff', dpi=600)
print("Figure_BUSCO_StackedBar.png generated.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# Standardize column names to remove leading/trailing whitespace
df_busco = df_busco.copy()
df_busco.columns = df_busco.columns.str.strip()

# Reshape from wide to long format for grouped plotting
busco_melt = df_busco.melt(
    id_vars=['Sample'],
    value_vars=['Single%', 'Dup%', 'Frag%', 'Miss%'],
    var_name='Category',
    value_name='Percentage'
)

busco_melt['Percentage'] = pd.to_numeric(busco_melt['Percentage'], errors='coerce')

order = ['Single%', 'Dup%', 'Frag%', 'Miss%']
busco_melt['Category'] = pd.Categorical(busco_melt['Category'], categories=order, ordered=True)

busco_colors = {
    'Single%': '#66BD63',
    'Dup%':    '#2C7BB6',
    'Frag%':   '#FFFFBF',
    'Miss%':   '#D73027'
}

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(10, 6))

# Draw boxplots without hue to avoid automatic legend conflicts,
# then apply category colors manually to each box patch
sns.boxplot(
    data=busco_melt,
    x='Category', y='Percentage',
    order=order,
    width=0.65,
    showfliers=False,
    linewidth=1.5,
    ax=ax
)

for patch, cat in zip(ax.patches[:len(order)], order):
    patch.set_facecolor(busco_colors[cat])
    patch.set_edgecolor('black')
    patch.set_alpha(0.9)

# Overlay individual data points colored by category
sns.stripplot(
    data=busco_melt,
    x='Category', y='Percentage',
    order=order,
    hue='Category',
    palette=busco_colors,
    dodge=False,
    jitter=True,
    size=4,
    alpha=0.45,
    linewidth=0,
    ax=ax
)

# Remove the automatic legend added by seaborn's hue parameter
leg = ax.get_legend()
if leg is not None:
    leg.remove()

# Construct a manual legend in the desired display order
handles = [mpatches.Patch(facecolor=busco_colors[cat], edgecolor='black', label=cat) for cat in order]
ax.legend(handles=handles, title='BUSCO Category', loc='upper right', frameon=True)

# Preserve the full axis span by avoiding spine trimming
sns.despine(ax=ax, trim=False, top=True, right=True)

ax.set_title('C. Gene Content Completeness (BUSCO)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('BUSCO Category', fontsize=12)
ax.set_ylabel('Percentage of Genes', fontsize=12)
ax.set_ylim(-1, 101)

fig.tight_layout()

fig.savefig('Figure_BUSCO_Boxplot_Fixed.png', dpi=300)
fig.savefig('Figure_BUSCO_Boxplot_Fixed.tiff', dpi=600)
print("Figure_BUSCO_Boxplot_Fixed.png and .tiff generated.")

In [91]:
df_busco_sorted.to_csv('busco_final.csv')